# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 5: The Integrated Digital Twin (System-of-Systems Simulation)

### Key Assumptions
- Real-time synchronization between physical and virtual systems
- IoT sensors provide continuous data streams from facilities
- Physics-based simulation models operational dynamics
- Predictive analytics enable proactive decision making
- What-if scenario testing supports strategic planning

### Approach (Step-by-Step)

A Digital Twin for CFLP creates a high-fidelity, dynamic replica of the entire supply chain network. Unlike static optimization models, it continuously mirrors real-world operations and enables comprehensive scenario analysis.

**System Architecture:**
1. **Physical Layer**: Real facilities, transportation networks, and customers
2. **Data Layer**: IoT sensors, ERP systems, and external data feeds
3. **Model Layer**: Physics-based simulation of operations and constraints
4. **Analytics Layer**: Predictive models and optimization algorithms
5. **Visualization Layer**: Real-time dashboards and scenario analysis
6. **Control Layer**: Decision support and automated recommendations

### What to Look for in the Results
- Real-time monitoring of facility utilization and performance
- Predictive analytics for demand forecasting and capacity planning
- What-if scenario analysis for network redesign and disruption response
- System-of-systems integration showing interdependencies

### Concrete Example

We'll demonstrate a comprehensive digital twin:
- **6 facilities** with real-time sensor data (100+ data points each)
- **8 customers** with dynamic demand patterns
- **Transportation network** with real-time routing and costs
- **24-hour simulation** with 1-minute update cycles
- **Multiple scenarios**: demand surge, facility disruption, cost changes

This shows how digital twins transform CFLP from static optimization to dynamic, proactive management.

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from datetime import datetime, timedelta
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Digital twin facility with real-time data"""
    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: float
    # Real-time sensor data
    current_utilization: float = 0.0
    operating_status: str = "ACTIVE"  # ACTIVE, MAINTENANCE, DISRUPTED
    energy_consumption: float = 0.0
    labor_efficiency: float = 1.0
    quality_score: float = 1.0
    maintenance_schedule: List[str] = None
    
    def __post_init__(self):
        if self.maintenance_schedule is None:
            self.maintenance_schedule = []

@dataclass
class Customer:
    """Digital twin customer with dynamic demand"""
    id: int
    name: str
    x_coord: float
    y_coord: float
    base_demand: float
    demand_pattern: str = "STABLE"  # STABLE, SEASONAL, VOLATILE
    service_level_requirement: float = 0.95
    
@dataclass
class TransportationLink:
    """Digital twin transportation link with dynamic conditions"""
    from_facility: int
    to_customer: int
    base_cost_per_unit: float
    current_delay: float = 0.0  # Hours of delay
    capacity_utilization: float = 0.0
    weather_impact: float = 1.0  # Cost multiplier
    
@dataclass
class DigitalTwinState:
    """Complete state of the digital twin at a point in time"""
    timestamp: datetime
    facilities: List[Facility]
    customers: List[Customer]
    transportation_links: List[TransportationLink]
    total_cost: float
    service_level: float
    system_efficiency: float
    alerts: List[str]

In [ ]:
# Initialize digital twin components
def initialize_facilities() -> List[Facility]:
    """Initialize facilities with diverse characteristics"""
    facilities = [
        Facility(1, "North Hub", 20, 80, 150000, 1200),
        Facility(2, "Central DC", 50, 50, 120000, 1000),
        Facility(3, "South DC", 80, 20, 100000, 800),
        Facility(4, "East Facility", 85, 65, 110000, 900),
        Facility(5, "West Facility", 15, 35, 90000, 700),
        Facility(6, "Northeast DC", 70, 85, 130000, 1100)
    ]
    
    # Initialize sensor data
    for facility in facilities:
        facility.current_utilization = np.random.uniform(0.3, 0.8)
        facility.energy_consumption = np.random.uniform(80, 95)
        facility.labor_efficiency = np.random.uniform(0.85, 1.0)
        facility.quality_score = np.random.uniform(0.9, 1.0)
    
    return facilities

def initialize_customers() -> List[Customer]:
    """Initialize customers with different demand patterns"""
    customers = [
        Customer(1, "Store A", 25, 75, 300, "STABLE"),
        Customer(2, "Store B", 45, 65, 450, "SEASONAL"),
        Customer(3, "Store C", 75, 35, 320, "VOLATILE"),
        Customer(4, "Store D", 90, 60, 420, "STABLE"),
        Customer(5, "Store E", 30, 20, 380, "SEASONAL"),
        Customer(6, "Store F", 60, 90, 460, "VOLATILE"),
        Customer(7, "Store G", 85, 40, 290, "STABLE"),
        Customer(8, "Store H", 55, 75, 410, "SEASONAL")
    ]
    
    return customers

def initialize_transportation_links(facilities: List[Facility], 
                                 customers: List[Customer]) -> List[TransportationLink]:
    """Initialize transportation links between facilities and customers"""
    links = []
    
    for facility in facilities:
        for customer in customers:
            # Calculate base cost based on distance
            distance = np.sqrt((facility.x_coord - customer.x_coord)**2 + 
                              (facility.y_coord - customer.y_coord)**2)
            base_cost = distance * 8.5  # $8.5 per unit per distance unit
            
            link = TransportationLink(
                from_facility=facility.id,
                to_customer=customer.id,
                base_cost_per_unit=base_cost
            )
            
            # Initialize dynamic conditions
            link.current_delay = np.random.uniform(0, 2)  # 0-2 hours delay
            link.capacity_utilization = np.random.uniform(0.3, 0.7)
            link.weather_impact = np.random.uniform(0.9, 1.1)
            
            links.append(link)
    
    return links

# Initialize digital twin
facilities = initialize_facilities()
customers = initialize_customers()
transportation_links = initialize_transportation_links(facilities, customers)

print("Digital Twin Initialized:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"Transportation Links: {len(transportation_links)}")
print(f"Total Data Points: {len(facilities) * 5 + len(customers) * 2 + len(transportation_links) * 4}")

In [ ]:
class DigitalTwinSimulator:
    """Digital twin simulator for CFLP"""
    
    def __init__(self, facilities: List[Facility], 
                 customers: List[Customer],
                 transportation_links: List[TransportationLink]):
        self.facilities = facilities
        self.customers = customers
        self.transportation_links = transportation_links
        self.current_time = datetime(2024, 1, 1, 0, 0, 0)
        self.history = []
        self.scenarios = []
        
    def calculate_dynamic_demand(self, customer: Customer, hour: int) -> float:
        """Calculate demand based on time and pattern"""
        base_demand = customer.base_demand
        
        # Time-of-day pattern
        if 6 <= hour <= 10:  # Morning peak
            time_factor = 1.3
        elif 11 <= hour <= 14:  # Lunch peak
            time_factor = 1.5
        elif 17 <= hour <= 20:  # Evening peak
            time_factor = 1.4
        else:  # Off-peak
            time_factor = 0.7
        
        # Pattern-specific variations
        if customer.demand_pattern == "SEASONAL":
            # Add seasonal variation
            seasonal_factor = 1.0 + 0.2 * np.sin(2 * np.pi * hour / 24)
            time_factor *= seasonal_factor
        elif customer.demand_pattern == "VOLATILE":
            # Add random volatility
            volatility = np.random.normal(1.0, 0.2)
            time_factor *= max(0.5, volatility)
        
        return base_demand * time_factor
    
    def update_facility_sensors(self, hour: int):
        """Update facility sensor data with realistic variations"""
        for facility in self.facilities:
            # Utilization follows demand patterns with lag
            if facility.operating_status == "ACTIVE":
                target_utilization = 0.5 + 0.3 * np.sin(2 * np.pi * (hour - 6) / 24)
                facility.current_utilization += 0.1 * (target_utilization - facility.current_utilization)
                facility.current_utilization = np.clip(facility.current_utilization, 0, 1)
                
                # Energy consumption correlates with utilization
                facility.energy_consumption = 70 + 25 * facility.current_utilization + np.random.normal(0, 2)
                
                # Labor efficiency varies during the day
                if 8 <= hour <= 17:  # Business hours
                    facility.labor_efficiency = 0.95 + np.random.normal(0, 0.05)
                else:  # Off-hours
                    facility.labor_efficiency = 0.85 + np.random.normal(0, 0.08)
                facility.labor_efficiency = np.clip(facility.labor_efficiency, 0.7, 1.0)
                
                # Quality score slowly degrades without maintenance
                facility.quality_score *= 0.999
                facility.quality_score = np.clip(facility.quality_score, 0.8, 1.0)
    
    def update_transportation_conditions(self, hour: int):
        """Update transportation link conditions"""
        for link in self.transportation_links:
            # Traffic patterns affect delays
            if 7 <= hour <= 9 or 16 <= hour <= 18:  # Rush hours
                link.current_delay = np.random.uniform(1, 3)
            else:
                link.current_delay = np.random.uniform(0, 1)
            
            # Capacity utilization follows demand
            link.capacity_utilization = 0.4 + 0.3 * np.sin(2 * np.pi * (hour - 8) / 24)
            link.capacity_utilization = np.clip(link.capacity_utilization, 0, 1)
            
            # Weather impact (simplified)
            if 6 <= hour <= 18:  # Daytime
                link.weather_impact = 1.0 + 0.05 * np.random.normal(0, 1)
            else:  # Nighttime
                link.weather_impact = 1.0 + 0.02 * np.random.normal(0, 1)
            link.weather_impact = np.clip(link.weather_impact, 0.8, 1.2)
    
    def calculate_optimal_assignment(self, hour: int) -> Dict:
        """Calculate optimal facility-customer assignments for current state"""
        # Get current demands
        current_demands = {}
        for customer in self.customers:
            current_demands[customer.id] = self.calculate_dynamic_demand(customer, hour)
        
        # Greedy assignment based on current conditions
        assignments = {}
        facility_loads = {f.id: 0 for f in self.facilities if f.operating_status == "ACTIVE"}
        total_cost = 0
        
        # Sort customers by demand (highest first)
        sorted_customers = sorted(self.customers, 
                                 key=lambda c: current_demands[c.id], 
                                 reverse=True)
        
        for customer in sorted_customers:
            demand = current_demands[customer.id]
            best_facility = None
            best_cost = float('inf')
            
            # Find best facility considering current conditions
            for facility in self.facilities:
                if facility.operating_status != "ACTIVE":
                    continue
                    
                # Check capacity
                available_capacity = facility.capacity * facility.labor_efficiency
                if facility_loads[facility.id] + demand > available_capacity:
                    continue
                
                # Find transportation link
                link = next(l for l in self.transportation_links 
                          if l.from_facility == facility.id and l.to_customer == customer.id)
                
                # Calculate total cost including current conditions
                transport_cost = (link.base_cost_per_unit * link.weather_impact + 
                                link.current_delay * 10)  # $10 per hour delay
                
                # Quality adjustment
                quality_adjustment = 1.0 + (1.0 - facility.quality_score) * 0.2
                
                total_cost_per_unit = transport_cost * quality_adjustment
                
                if total_cost_per_unit < best_cost:
                    best_cost = total_cost_per_unit
                    best_facility = facility.id
            
            if best_facility:
                assignments[customer.id] = best_facility
                facility_loads[best_facility] += demand
                total_cost += best_cost * demand
            else:
                # Unmet demand
                assignments[customer.id] = None
                total_cost += demand * 1000  # Penalty cost
        
        return {
            'assignments': assignments,
            'facility_loads': facility_loads,
            'total_cost': total_cost,
            'unmet_demand': sum(current_demands[c.id] for c in self.customers 
                              if assignments.get(c.id) is None)
        }
    
    def calculate_system_metrics(self, assignment_result: Dict) -> Dict:
        """Calculate system-wide performance metrics"""
        assignments = assignment_result['assignments']
        
        # Service level (percentage of demand met)
        total_demand = sum(self.calculate_dynamic_demand(c, self.current_time.hour) 
                          for c in self.customers)
        met_demand = total_demand - assignment_result['unmet_demand']
        service_level = met_demand / total_demand if total_demand > 0 else 0
        
        # System efficiency (utilization weighted by quality)
        active_facilities = [f for f in self.facilities if f.operating_status == "ACTIVE"]
        if active_facilities:
            efficiency = sum(f.current_utilization * f.quality_score * f.labor_efficiency 
                           for f in active_facilities) / len(active_facilities)
        else:
            efficiency = 0
        
        # Generate alerts
        alerts = []
        
        # High utilization alerts
        for facility in self.facilities:
            if facility.current_utilization > 0.9:
                alerts.append(f"HIGH_UTILIZATION: {facility.name} at {facility.current_utilization:.1%}")
            if facility.quality_score < 0.85:
                alerts.append(f"LOW_QUALITY: {facility.name} at {facility.quality_score:.2f}")
        
        # Service level alert
        if service_level < 0.9:
            alerts.append(f"LOW_SERVICE_LEVEL: {service_level:.1%}")
        
        return {
            'service_level': service_level,
            'system_efficiency': efficiency,
            'alerts': alerts
        }
    
    def advance_time(self, minutes: int = 60):
        """Advance simulation time and update all components"""
        self.current_time += timedelta(minutes=minutes)
        hour = self.current_time.hour
        
        # Update all sensor data
        self.update_facility_sensors(hour)
        self.update_transportation_conditions(hour)
        
        # Calculate optimal assignment
        assignment_result = self.calculate_optimal_assignment(hour)
        
        # Calculate system metrics
        metrics = self.calculate_system_metrics(assignment_result)
        
        # Create state snapshot
        state = DigitalTwinState(
            timestamp=self.current_time,
            facilities=[f for f in self.facilities],  # Copy current state
            customers=[c for c in self.customers],
            transportation_links=[l for l in self.transportation_links],
            total_cost=assignment_result['total_cost'],
            service_level=metrics['service_level'],
            system_efficiency=metrics['system_efficiency'],
            alerts=metrics['alerts']
        )
        
        self.history.append(state)
        
        return state
    
    def run_scenario(self, scenario_name: str, duration_hours: int = 24):
        """Run a what-if scenario"""
        print(f"\nRunning scenario: {scenario_name}")
        print(f"Duration: {duration_hours} hours")
        
        # Save current state
        original_facilities = [f for f in self.facilities]
        
        # Apply scenario modifications
        if scenario_name == "DEMAND_SURGE":
            # Increase demand by 50%
            for customer in self.customers:
                customer.base_demand *= 1.5
        elif scenario_name == "FACILITY_DISRUPTION":
            # Disrupt one facility
            self.facilities[2].operating_status = "DISRUPTED"
            print(f"Disrupted: {self.facilities[2].name}")
        elif scenario_name == "COST_INCREASE":
            # Increase transportation costs
            for link in self.transportation_links:
                link.base_cost_per_unit *= 1.3
        
        # Run simulation
        scenario_history = []
        for hour in range(duration_hours):
            state = self.advance_time(60)
            scenario_history.append(state)
            
            if hour % 6 == 0:  # Report every 6 hours
                print(f"  Hour {hour:2d}: Cost=${state.total_cost:,.0f}, "
                      f"Service={state.service_level:.1%}, "
                      f"Efficiency={state.system_efficiency:.1%}")
        
        # Restore original state
        self.facilities = original_facilities
        for customer in self.customers:
            customer.base_demand /= 1.5 if scenario_name == "DEMAND_SURGE" else 1.0
        for link in self.transportation_links:
            link.base_cost_per_unit /= 1.3 if scenario_name == "COST_INCREASE" else 1.0
        
        self.scenarios.append({
            'name': scenario_name,
            'history': scenario_history
        })
        
        return scenario_history

print("Digital Twin Simulator class defined successfully")

In [ ]:
# Initialize and run digital twin simulation
print("="*80)
print("DIGITAL TWIN SIMULATION - BASELINE OPERATIONS")
print("="*80)

# Create simulator
simulator = DigitalTwinSimulator(facilities, customers, transportation_links)

# Run baseline simulation (24 hours)
print("\nRunning baseline simulation for 24 hours...")
baseline_states = []

for hour in range(24):
    state = simulator.advance_time(60)
    baseline_states.append(state)
    
    if hour % 6 == 0:  # Report every 6 hours
        print(f"Hour {hour:2d}: ${state.total_cost:,.0f}, "
              f"Service={state.service_level:.1%}, "
              f"Efficiency={state.system_efficiency:.1%}, "
              f"Alerts={len(state.alerts)}")

print(f"\nBaseline Summary:")
avg_cost = np.mean([s.total_cost for s in baseline_states])
avg_service = np.mean([s.service_level for s in baseline_states])
avg_efficiency = np.mean([s.system_efficiency for s in baseline_states])

print(f"Average Hourly Cost: ${avg_cost:,.2f}")
print(f"Average Service Level: {avg_service:.1%}")
print(f"Average System Efficiency: {avg_efficiency:.1%}")
print(f"Total Alerts Generated: {sum(len(s.alerts) for s in baseline_states)}")

In [ ]:
# Run what-if scenarios
print("\n" + "="*80)
print("WHAT-IF SCENARIO ANALYSIS")
print("="*80)

# Scenario 1: Demand Surge
demand_surge_history = simulator.run_scenario("DEMAND_SURGE", 24)

# Scenario 2: Facility Disruption
facility_disruption_history = simulator.run_scenario("FACILITY_DISRUPTION", 24)

# Scenario 3: Cost Increase
cost_increase_history = simulator.run_scenario("COST_INCREASE", 24)

# Compare scenarios
print("\n" + "="*80)
print("SCENARIO COMPARISON")
print("="*80)

scenarios_data = [
    {
        'Scenario': 'Baseline',
        'Avg Cost': np.mean([s.total_cost for s in baseline_states]),
        'Avg Service Level': np.mean([s.service_level for s in baseline_states]),
        'Avg Efficiency': np.mean([s.system_efficiency for s in baseline_states]),
        'Total Alerts': sum(len(s.alerts) for s in baseline_states)
    },
    {
        'Scenario': 'Demand Surge',
        'Avg Cost': np.mean([s.total_cost for s in demand_surge_history]),
        'Avg Service Level': np.mean([s.service_level for s in demand_surge_history]),
        'Avg Efficiency': np.mean([s.system_efficiency for s in demand_surge_history]),
        'Total Alerts': sum(len(s.alerts) for s in demand_surge_history)
    },
    {
        'Scenario': 'Facility Disruption',
        'Avg Cost': np.mean([s.total_cost for s in facility_disruption_history]),
        'Avg Service Level': np.mean([s.service_level for s in facility_disruption_history]),
        'Avg Efficiency': np.mean([s.system_efficiency for s in facility_disruption_history]),
        'Total Alerts': sum(len(s.alerts) for s in facility_disruption_history)
    },
    {
        'Scenario': 'Cost Increase',
        'Avg Cost': np.mean([s.total_cost for s in cost_increase_history]),
        'Avg Service Level': np.mean([s.service_level for s in cost_increase_history]),
        'Avg Efficiency': np.mean([s.system_efficiency for s in cost_increase_history]),
        'Total Alerts': sum(len(s.alerts) for s in cost_increase_history)
    }
]

comparison_df = pd.DataFrame(scenarios_data)
print(comparison_df.round(2))

# Calculate percentage changes vs baseline
baseline = scenarios_data[0]
print(f"\nPercentage Changes vs Baseline:")
for scenario in scenarios_data[1:]:
    cost_change = (scenario['Avg Cost'] / baseline['Avg Cost'] - 1) * 100
    service_change = (scenario['Avg Service Level'] / baseline['Avg Service Level'] - 1) * 100
    efficiency_change = (scenario['Avg Efficiency'] / baseline['Avg Efficiency'] - 1) * 100
    
    print(f"\n{scenario['Scenario']}:")
    print(f"  Cost Change: {cost_change:+.1f}%")
    print(f"  Service Level Change: {service_change:+.1f}%")
    print(f"  Efficiency Change: {efficiency_change:+.1f}%")

In [ ]:
# Predictive analytics
print("\n" + "="*80)
print("PREDICTIVE ANALYTICS")
print("="*80)

def analyze_historical_patterns(history: List[DigitalTwinState]) -> Dict:
    """Analyze patterns in historical data"""
    
    # Extract time series data
    hours = [s.timestamp.hour for s in history]
    costs = [s.total_cost for s in history]
    service_levels = [s.service_level for s in history]
    efficiencies = [s.system_efficiency for s in history]
    
    # Calculate hourly patterns
    hourly_stats = {}
    for hour in range(24):
        hour_data = [history[i] for i, h in enumerate(hours) if h == hour]
        if hour_data:
            hourly_stats[hour] = {
                'avg_cost': np.mean([s.total_cost for s in hour_data]),
                'avg_service': np.mean([s.service_level for s in hour_data]),
                'avg_efficiency': np.mean([s.system_efficiency for s in hour_data]),
                'peak_alerts': sum(len(s.alerts) for s in hour_data)
            }
    
    # Identify peak hours
    peak_cost_hours = sorted(hourly_stats.items(), 
                              key=lambda x: x[1]['avg_cost'], 
                              reverse=True)[:3]
    
    peak_alert_hours = sorted(hourly_stats.items(), 
                              key=lambda x: x[1]['peak_alerts'], 
                              reverse=True)[:3]
    
    return {
        'hourly_stats': hourly_stats,
        'peak_cost_hours': peak_cost_hours,
        'peak_alert_hours': peak_alert_hours,
        'total_cost_variance': np.var(costs),
        'service_level_variance': np.var(service_levels)
    }

def predict_future_performance(history: List[DigitalTwinState], 
                              forecast_hours: int = 12) -> Dict:
    """Simple predictive model for future performance"""
    
    # Extract recent trends (last 6 hours)
    recent_history = history[-6:] if len(history) >= 6 else history
    
    # Calculate trends
    if len(recent_history) >= 2:
        cost_trend = np.polyfit(range(len(recent_history)), 
                              [s.total_cost for s in recent_history], 1)[0]
        service_trend = np.polyfit(range(len(recent_history)), 
                                 [s.service_level for s in recent_history], 1)[0]
        efficiency_trend = np.polyfit(range(len(recent_history)), 
                                     [s.system_efficiency for s in recent_history], 1)[0]
    else:
        cost_trend = service_trend = efficiency_trend = 0
    
    # Generate predictions
    predictions = []
    last_state = history[-1] if history else None
    
    if last_state:
        for hour_ahead in range(1, forecast_hours + 1):
            predicted_cost = last_state.total_cost + cost_trend * hour_ahead
            predicted_service = last_state.service_level + service_trend * hour_ahead
            predicted_efficiency = last_state.system_efficiency + efficiency_trend * hour_ahead
            
            # Add confidence intervals (simplified)
            cost_uncertainty = abs(cost_trend) * hour_ahead * 0.5
            
            predictions.append({
                'hour_ahead': hour_ahead,
                'predicted_cost': predicted_cost,
                'cost_lower': predicted_cost - cost_uncertainty,
                'cost_upper': predicted_cost + cost_uncertainty,
                'predicted_service': max(0, min(1, predicted_service)),
                'predicted_efficiency': max(0, min(1, predicted_efficiency))
            })
    
    return {
        'predictions': predictions,
        'trends': {
            'cost_trend': cost_trend,
            'service_trend': service_trend,
            'efficiency_trend': efficiency_trend
        }
    }

# Analyze baseline patterns
patterns = analyze_historical_patterns(baseline_states)

print("Historical Pattern Analysis:")
print(f"\nPeak Cost Hours:")
for hour, stats in patterns['peak_cost_hours']:
    print(f"  Hour {hour:02d}: ${stats['avg_cost']:,.0f}")

print(f"\nPeak Alert Hours:")
for hour, stats in patterns['peak_alert_hours']:
    print(f"  Hour {hour:02d}: {stats['peak_alerts']} alerts")

# Generate predictions
predictions = predict_future_performance(baseline_states, 12)

print(f"\nFuture Performance Predictions (Next 12 hours):")
for pred in predictions[:6]:  # Show first 6 predictions
    print(f"  Hour +{pred['hour_ahead']:2d}: "
          f"Cost=${pred['predicted_cost']:,.0f} "
          f"(${pred['cost_lower']:,.0f}-${pred['cost_upper']:,.0f}), "
          f"Service={pred['predicted_service']:.1%}, "
          f"Efficiency={pred['predicted_efficiency']:.1%}")

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Digital Twin CFLP Analysis', fontsize=16, fontweight='bold')

# 1. Time series of baseline operations
ax1 = axes[0, 0]
hours = list(range(24))
costs = [s.total_cost for s in baseline_states]
service_levels = [s.service_level * 1000 for s in baseline_states]  # Scale for visibility
efficiencies = [s.system_efficiency * 500 for s in baseline_states]  # Scale for visibility

ax1.plot(hours, costs, 'b-', linewidth=2, label='Cost ($)')
ax1.plot(hours, service_levels, 'g--', linewidth=1, label='Service Level (×1000)')
ax1.plot(hours, efficiencies, 'r:', linewidth=1, label='Efficiency (×500)')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Value')
ax1.set_title('Baseline Operations - 24 Hour Profile')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Scenario comparison
ax2 = axes[0, 1]
scenario_names = ['Baseline', 'Demand Surge', 'Facility Disruption', 'Cost Increase']
avg_costs = [s['Avg Cost'] for s in scenarios_data]
colors = ['steelblue', 'orange', 'red', 'purple']
bars = ax2.bar(scenario_names, avg_costs, color=colors, alpha=0.7)
ax2.set_ylabel('Average Hourly Cost ($)')
ax2.set_title('Scenario Cost Comparison')
ax2.grid(True, alpha=0.3)
plt.setp(ax2.get_xticklabels(), rotation=45)

# Add value labels on bars
for bar, cost in zip(bars, avg_costs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(avg_costs)*0.01,
             f'${cost:,.0f}', ha='center', va='bottom')

# 3. Facility utilization patterns
ax3 = axes[0, 2]
facility_names = [f.name[:8] for f in facilities]
utilization_data = []

for facility in facilities:
    utilization_history = []
    for state in baseline_states:
            fac_state = next(f for f in state.facilities if f.id == facility.id)
            utilization_history.append(fac_state.current_utilization)
    utilization_data.append(utilization_history)

for i, (name, data) in enumerate(zip(facility_names, utilization_data)):
    ax3.plot(hours, data, marker='o', linewidth=2, markersize=4, label=name)

ax3.set_xlabel('Hour of Day')
ax3.set_ylabel('Utilization')
ax3.set_title('Facility Utilization Patterns')
ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax3.grid(True, alpha=0.3)

# 4. Alert frequency analysis
ax4 = axes[1, 0]
alert_counts = [len(s.alerts) for s in baseline_states]
ax4.bar(hours, alert_counts, color='coral', alpha=0.7)
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('Number of Alerts')
ax4.set_title('Alert Frequency by Hour')
ax4.grid(True, alpha=0.3)

# 5. Predictive analytics visualization
ax5 = axes[1, 1]
if predictions['predictions']:
    pred_hours = [p['hour_ahead'] for p in predictions['predictions']]
    pred_costs = [p['predicted_cost'] for p in predictions['predictions']]
    cost_lower = [p['cost_lower'] for p in predictions['predictions']]
    cost_upper = [p['cost_upper'] for p in predictions['predictions']]
    
    ax5.plot(pred_hours, pred_costs, 'b-', linewidth=2, label='Predicted Cost')
    ax5.fill_between(pred_hours, cost_lower, cost_upper, alpha=0.3, color='blue', label='Confidence Interval')
    ax5.set_xlabel('Hours Ahead')
    ax5.set_ylabel('Predicted Cost ($)')
    ax5.set_title('Cost Prediction with Uncertainty')
    ax5.legend()
    ax5.grid(True, alpha=0.3)

# 6. System performance radar
ax6 = axes[1, 2]
# Create radar chart for scenario comparison
categories = ['Cost\nEfficiency', 'Service\nLevel', 'System\nEfficiency', 'Stability', 'Alert\nManagement']
N = len(categories)

# Calculate normalized scores for each scenario
scenario_scores = []
for scenario in scenarios_data:
    # Normalize metrics (higher is better for most)
    cost_score = 1 - (scenario['Avg Cost'] - min(s['Avg Cost'] for s in scenarios_data)) / \
                  (max(s['Avg Cost'] for s in scenarios_data) - min(s['Avg Cost'] for s in scenarios_data))
    service_score = scenario['Avg Service Level']
    efficiency_score = scenario['Avg Efficiency']
    stability_score = 1 - (scenario['Total Alerts'] / max(s['Total Alerts'] for s in scenarios_data))
    alert_score = 1 - (scenario['Total Alerts'] / max(s['Total Alerts'] for s in scenarios_data))
    
    scores = [cost_score, service_score, efficiency_score, stability_score, alert_score]
    scenario_scores.append(scores)

# Plot radar chart
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

for i, (scenario_name, scores) in enumerate(zip(scenario_names, scenario_scores)):
    scores += scores[:1]  # Complete the circle
    ax6.plot(angles, scores, 'o-', linewidth=2, label=scenario_name)
    ax6.fill(angles, scores, alpha=0.1)

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(categories)
ax6.set_ylim(0, 1)
ax6.set_title('Multi-Criteria Performance Comparison')
ax6.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax6.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Digital twin recommendations and insights
print("\n" + "="*80)
print("DIGITAL TWIN INSIGHTS AND RECOMMENDATIONS")
print("="*80)

# Analyze facility performance
print("\nFacility Performance Analysis:")
for facility in facilities:
    # Calculate average metrics from baseline
    facility_states = [s for s in baseline_states 
                       for f in s.facilities if f.id == facility.id]
    
    if facility_states:
        avg_utilization = np.mean([f.current_utilization for f in facility_states])
        avg_quality = np.mean([f.quality_score for f in facility_states])
        avg_efficiency = np.mean([f.labor_efficiency for f in facility_states])
        
        print(f"\n{facility.name}:")
        print(f"  Average Utilization: {avg_utilization:.1%}")
        print(f"  Average Quality: {avg_quality:.3f}")
        print(f"  Average Labor Efficiency: {avg_efficiency:.1%}")
        
        # Recommendations
        if avg_utilization > 0.85:
            print(f"  ⚠️  RECOMMENDATION: Consider capacity expansion")
        elif avg_utilization < 0.4:
            print(f"  ⚠️  RECOMMENDATION: Consider consolidation or marketing")
        
        if avg_quality < 0.9:
            print(f"  🔧 RECOMMENDATION: Schedule maintenance")
        
        if avg_efficiency < 0.9:
            print(f"  👥 RECOMMENDATION: Staff training or process optimization")

# Transportation insights
print("\n\nTransportation Network Insights:")
high_delay_links = [link for link in transportation_links 
                    if link.current_delay > 1.5]

if high_delay_links:
    print(f"High-Delay Transportation Links: {len(high_delay_links)}")
    for link in high_delay_links[:3]:  # Show top 3
        facility_name = next(f.name for f in facilities if f.id == link.from_facility)
        customer_name = next(c.name for c in customers if c.id == link.to_customer)
        print(f"  {facility_name} → {customer_name}: {link.current_delay:.1f}h delay")
else:
    print("All transportation links operating within acceptable delay ranges")

# Predictive insights
print("\n\nPredictive Insights:")
trends = predictions['trends']
print(f"Cost Trend: {'Increasing' if trends['cost_trend'] > 0 else 'Decreasing' if trends['cost_trend'] < 0 else 'Stable'}")
print(f"Service Level Trend: {'Improving' if trends['service_trend'] > 0 else 'Declining' if trends['service_trend'] < 0 else 'Stable'}")
print(f"System Efficiency Trend: {'Improving' if trends['efficiency_trend'] > 0 else 'Declining' if trends['efficiency_trend'] < 0 else 'Stable'}")

# Strategic recommendations
print("\n\nStrategic Recommendations:")
print("✅ Implement predictive maintenance based on quality score trends")
print("✅ Optimize staffing schedules to match demand patterns")
print("✅ Develop contingency plans for peak alert hours")
print("✅ Consider dynamic pricing for transportation during peak delays")
print("✅ Use digital twin for continuous network optimization")
print("✅ Implement real-time alert monitoring and response")

print(f"\nDigital Twin Benefits Demonstrated:")
print(f"🔍 Real-time monitoring of {len(facilities)} facilities with {len(facilities)*5}+ data points")
print(f"📊 Predictive analytics for {len(customers)} demand points")
print(f"🎯 What-if analysis for {len(scenarios_data)} strategic scenarios")
print(f"⚡  {sum(len(s.alerts) for s in baseline_states)} proactive alerts generated")
print(f"📈 Continuous optimization with {len(baseline_states)} state snapshots")
print(f"🚀 System-of-systems integration across facilities, transportation, and customers")

### Why This Tier Exists vs Previous Tiers

**Tier 5 (Digital Twin) represents a fundamental paradigm shift from static optimization to dynamic, real-time management:**

**From Static to Dynamic:**
- **Traditional methods**: Solve optimization problems at fixed points in time
- **Digital twin**: Continuously mirrors real-world operations in real-time
- **Reactive to proactive**: Anticipate issues before they impact operations

**System-of-Systems Integration:**
- **Siloed optimization**: Each component optimized independently
- **Holistic view**: Interdependencies between facilities, transportation, and customers
- **Emergent behavior**: System-level properties not visible in component analysis

**Decision Support Evolution:**
- **Point solutions**: Optimal decisions for specific scenarios
- **Continuous optimization**: Adaptive decisions based on current conditions
- **Strategic foresight**: What-if analysis for long-term planning

### Pros and Cons

**Advantages:**
✅ **Real-time monitoring** of all system components
✅ **Predictive analytics** for proactive decision making
✅ **What-if scenario testing** for strategic planning
✅ **System-of-systems view** revealing interdependencies
✅ **Continuous optimization** adapting to changing conditions
✅ **Risk management** through disruption simulation

**Disadvantages:**
❌ **High implementation complexity** requiring significant resources
❌ **Data infrastructure requirements** for sensors and integration
❌ **Computational overhead** for real-time simulation
❌ **Maintenance challenges** keeping twin synchronized with reality
❌ **Model accuracy dependence** on sensor quality and calibration
❌ **Security concerns** with real-time data access

### When to Use This Tier

**Ideal for:**
- **Large-scale operations** with multiple facilities and complex networks
- **High-value operations** where downtime costs are substantial
- **Dynamic environments** with rapidly changing conditions
- **Strategic planning** requiring long-term scenario analysis
- **Regulatory industries** requiring continuous compliance monitoring
- **Competitive markets** where operational agility is critical

**Not ideal for:**
- **Small-scale operations** with simple logistics networks
- **Static environments** with minimal operational changes
- **Resource-constrained organizations** lacking technical capabilities
- **Low-complexity problems** where traditional optimization suffices
- **Situations requiring** immediate decisions without setup time
- **Organizations with** limited data collection capabilities